In [11]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import pearsonr
import warnings
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

In [12]:
# =========================
# Configuration
# =========================
class Config:
    TRAIN_PATH       = "C:/Users/arnov/Desktop/Projet/Kaggel/dw crypto/drw-crypto-market-prediction/train.parquet"
    TEST_PATH        = "C:/Users/arnov/Desktop/Projet/Kaggel/dw crypto/drw-crypto-market-prediction/test.parquet"
    SUBMISSION_PATH  = "C:/Users/arnov/Desktop/Projet/Kaggel/dw crypto/drw-crypto-market-prediction/sample_submission.csv"
    
    FEATURES = [
        "X344", "X598", "X385",  "X603",  "X674",
        "X415", "X345", "X137", "X174", "X302", "X178", "X532", "X168", "X612",
        "bid_qty", "ask_qty", "buy_qty", "sell_qty", "volume",  "X421", "X333"
    ]

    
    LABEL_COLUMN     = "label"
    N_FOLDS          = 3
    RANDOM_STATE     = 42

# Hyperparameters for XGBoost and LightGBM
XGB_PARAMS = {
    "tree_method": "hist",
    "device": "gpu",
    "colsample_bylevel": 0.4778,
    "colsample_bynode": 0.3628,
    "colsample_bytree": 0.7107,
    "gamma": 1.7095,
    "learning_rate": 0.02213,
    "max_depth": 20,
    "max_leaves": 12,
    "min_child_weight": 16,
    "n_estimators": 1667,
    "subsample": 0.06567,
    "reg_alpha": 39.3524,
    "reg_lambda": 75.4484,
    "verbosity": 0,
    "random_state": Config.RANDOM_STATE,
    "n_jobs": -1,
    "verbose": False,
}

LGBM_PARAMS = {
    "boosting_type": "gbdt",
    "device": "cpu",
    "n_jobs": -1,
    "verbose": -1,
    "random_state": Config.RANDOM_STATE,
    "colsample_bytree": 0.5039,
    "learning_rate": 0.01260,
    "min_child_samples": 20,
    "min_child_weight": 0.1146,
    "n_estimators": 915,
    "num_leaves": 145,
    "reg_alpha": 19.2447,
    "reg_lambda": 55.5046,
    "subsample": 0.9709,
    "max_depth": 9
}

# Define new learners
LEARNERS = [
    {"name": "xgb", "Estimator": XGBRegressor, "params": XGB_PARAMS, "need_scale": False},
    {"name" : 'lgbm', "Estimator" : LGBMRegressor, 'params' : LGBM_PARAMS, "need_scale": False}
]

In [13]:
# =========================
# Utility Functions
# =========================
def create_time_decay_weights(n: int, decay: float = 0.95) -> np.ndarray:
    positions = np.arange(n)
    normalized = positions / (n - 1)
    weights = decay ** (1.0 - normalized)
    return weights * n / weights.sum()

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # === NEW FEATURES ===
    df['volume_weighted_sell'] = df['sell_qty'] * df['volume']
    df['buy_sell_ratio'] = df['buy_qty'] / (df['sell_qty'] + 1e-8)
    df['selling_pressure'] = df['sell_qty'] / (df['volume'] + 1e-8)
    df['effective_spread_proxy'] = np.abs(df['buy_qty'] - df['sell_qty']) / (df['volume'] + 1e-8)
    df['log_volume'] = np.log1p(df['volume'])
    df['bid_ask_imbalance'] = (df['bid_qty'] - df['ask_qty']) / (df['bid_qty'] + df['ask_qty'] + 1e-8)
    df['order_flow_imbalance'] = (df['buy_qty'] - df['sell_qty']) / (df['buy_qty'] + df['sell_qty'] + 1e-8)
    df['liquidity_ratio'] = (df['bid_qty'] + df['ask_qty']) / (df['volume'] + 1e-8)

    return df

def load_data():
    # Charger sans filtrer pour éviter les erreurs de colonnes manquantes
    train_df = pd.read_parquet(Config.TRAIN_PATH)
    test_df = pd.read_parquet(Config.TEST_PATH)
    submission_df = pd.read_csv(Config.SUBMISSION_PATH)

    # Vérifier que les colonnes attendues sont présentes dans les données brutes
    required_train_cols = Config.FEATURES + [Config.LABEL_COLUMN]
    missing_train_cols = [col for col in required_train_cols if col not in train_df.columns]
    missing_test_cols = [col for col in Config.FEATURES if col not in test_df.columns]

    if missing_train_cols:
        raise ValueError(f"Colonnes manquantes dans le train.parquet: {missing_train_cols}")
    if missing_test_cols:
        raise ValueError(f"Colonnes manquantes dans le test.parquet: {missing_test_cols}")

    # Sélectionner uniquement les colonnes nécessaires (features + label)
    train_df = train_df[required_train_cols]
    test_df = test_df[Config.FEATURES]

    print(f"Loaded data - Train: {train_df.shape}, Test: {test_df.shape}, Submission: {submission_df.shape}")

    # Feature engineering (implémenter cette fonction ailleurs)
    train_df = feature_engineering(train_df)
    test_df = feature_engineering(test_df)

    # Nettoyer NaN : supprimer lignes dans train, remplir test par 0
    train_df = train_df.dropna().reset_index(drop=True)
    test_df = test_df.fillna(0)

    # Mettre à jour dynamiquement la liste des features dans Config
    engineered_features = [col for col in train_df.columns if col != Config.LABEL_COLUMN]
    setattr(Config, "FEATURES", engineered_features)

    print(f"Processed data - Train: {train_df.shape}, Test: {test_df.shape}, Submission: {submission_df.shape}")

    return train_df, test_df, submission_df

def get_model_slices(n_samples: int):
    return [
        {"name": "full_data", "cutoff": 0},
        {"name": "last_75pct", "cutoff": int(0.25 * n_samples)},
        {"name": "last_50pct", "cutoff": int(0.50 * n_samples)}
    ]

# =========================
# Training and Evaluation
# =========================
def get_model_slices(n_samples: int):
    """Define different data slices for training"""
    return [
        {"name": "full_data", "cutoff": 0},
        {"name": "last_75pct", "cutoff": int(0.25 * n_samples)},
        {"name": "last_50pct", "cutoff": int(0.50 * n_samples)},
    ]

def train_single_model(X_train, y_train, X_valid, y_valid, X_test, learner, sample_weights=None):
    """Train a single model with appropriate scaling if needed"""
    if learner["need_scale"]:
        scaler = RobustScaler()  # More robust to outliers than StandardScaler
        X_train_scaled = scaler.fit_transform(X_train)
        X_valid_scaled = scaler.transform(X_valid)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_scaled = X_train
        X_valid_scaled = X_valid
        X_test_scaled = X_test
    
    model = learner["Estimator"](**learner["params"])
    
    # Handle different model training approaches
    if learner["name"] in ["xgb_baseline", "lgbm"]:
        if learner["name"] == "xgb_baseline":
            model.fit(X_train_scaled, y_train, sample_weight=sample_weights, 
                     eval_set=[(X_valid_scaled, y_valid)], verbose=False)
        else:  # LightGBM
            model.fit(X_train_scaled, y_train, sample_weight=sample_weights,
                     eval_set=[(X_valid_scaled, y_valid)], callbacks=[])
    elif learner["name"] in ["huber", "lasso", "elasticnet"]:
        model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
    else:
        # RANSAC, TheilSen, PLS don't support sample weights
        model.fit(X_train_scaled, y_train)
    
    valid_pred = model.predict(X_valid_scaled)
    test_pred = model.predict(X_test_scaled)
    
    return valid_pred, test_pred

def train_and_evaluate(train_df, test_df):
    """Train all models with cross-validation"""
    n_samples = len(train_df)
    model_slices = get_model_slices(n_samples)
    
    # Initialize prediction dictionaries
    oof_preds = {
        learner["name"]: {s["name"]: np.zeros(n_samples) for s in model_slices}
        for learner in LEARNERS
    }
    test_preds = {
        learner["name"]: {s["name"]: np.zeros(len(test_df)) for s in model_slices}
        for learner in LEARNERS
    }
    
    full_weights = create_time_decay_weights(n_samples)
    kf = KFold(n_splits=Config.N_FOLDS, shuffle=False)
    
    for fold, (train_idx, valid_idx) in enumerate(kf.split(train_df), start=1):
        print(f"\n--- Fold {fold}/{Config.N_FOLDS} ---")
        X_valid = train_df.iloc[valid_idx][Config.FEATURES]
        y_valid = train_df.iloc[valid_idx][Config.LABEL_COLUMN]
        X_test = test_df[Config.FEATURES]
        
        for s in model_slices:
            cutoff = s["cutoff"]
            slice_name = s["name"]
            subset = train_df.iloc[cutoff:].reset_index(drop=True)
            rel_idx = train_idx[train_idx >= cutoff] - cutoff
            
            if len(rel_idx) == 0:
                continue
                
            X_train = subset.iloc[rel_idx][Config.FEATURES]
            y_train = subset.iloc[rel_idx][Config.LABEL_COLUMN]
            sw = create_time_decay_weights(len(subset))[rel_idx] if cutoff > 0 else full_weights[train_idx]
            
            print(f"  Training slice: {slice_name}, samples: {len(X_train)}")
            
            for learner in LEARNERS:
                try:
                    valid_pred, test_pred = train_single_model(
                        X_train, y_train, X_valid, y_valid, X_test, learner, sw
                    )
                    
                    # Store OOF predictions
                    mask = valid_idx >= cutoff
                    if mask.any():
                        idxs = valid_idx[mask]
                        X_valid_subset = train_df.iloc[idxs][Config.FEATURES]
                        if learner["need_scale"]:
                            scaler = RobustScaler()
                            scaler.fit(X_train)
                            valid_pred_subset = learner["Estimator"](**learner["params"]).fit(
                                scaler.transform(X_train), y_train
                            ).predict(scaler.transform(X_valid_subset))
                            oof_preds[learner["name"]][slice_name][idxs] = valid_pred_subset
                        else:
                            oof_preds[learner["name"]][slice_name][idxs] = valid_pred[mask]
                    
                    if cutoff > 0 and (~mask).any():
                        oof_preds[learner["name"]][slice_name][valid_idx[~mask]] = \
                            oof_preds[learner["name"]]["full_data"][valid_idx[~mask]]
                    
                    test_preds[learner["name"]][slice_name] += test_pred
                    
                except Exception as e:
                    print(f"    Error training {learner['name']}: {str(e)}")
                    continue
    
    # Normalize test predictions
    for learner_name in test_preds:
        for slice_name in test_preds[learner_name]:
            test_preds[learner_name][slice_name] /= Config.N_FOLDS
    
    return oof_preds, test_preds, model_slices

In [14]:
# =========================
# Ensemble & Submission
# =========================
def ensemble_and_submit(train_df, oof_preds, test_preds, submission_df):
    learner_ensembles = {}
    for learner_name in oof_preds:
        scores = {s: pearsonr(train_df[Config.LABEL_COLUMN], oof_preds[learner_name][s])[0]
                  for s in oof_preds[learner_name]}
        total_score = sum(scores.values())

        oof_simple = np.mean(list(oof_preds[learner_name].values()), axis=0)
        test_simple = np.mean(list(test_preds[learner_name].values()), axis=0)
        score_simple = pearsonr(train_df[Config.LABEL_COLUMN], oof_simple)[0]

        oof_weighted = sum(scores[s] / total_score * oof_preds[learner_name][s] for s in scores)
        test_weighted = sum(scores[s] / total_score * test_preds[learner_name][s] for s in scores)
        score_weighted = pearsonr(train_df[Config.LABEL_COLUMN], oof_weighted)[0]

        print(f"\n{learner_name.upper()} Simple Ensemble Pearson:   {score_simple:.4f}")
        print(f"{learner_name.upper()} Weighted Ensemble Pearson: {score_weighted:.4f}")

        learner_ensembles[learner_name] = {
            "oof_simple": oof_simple,
            "test_simple": test_simple
        }

    final_oof = np.mean([le["oof_simple"] for le in learner_ensembles.values()], axis=0)
    final_test = np.mean([le["test_simple"] for le in learner_ensembles.values()], axis=0)
    final_score = pearsonr(train_df[Config.LABEL_COLUMN], final_oof)[0]

    print(f"\nFINAL ensemble across learners Pearson: {final_score:.4f}")

    submission_df["prediction"] = final_test
    submission_df.to_csv("submission.csv", index=False)
    print("Saved: submission.csv")

In [15]:
# =========================
# Main Execution
# =========================
train_df, test_df, submission_df = load_data()
train_df

Loaded data - Train: (525886, 22), Test: (538150, 21), Submission: (538150, 2)
Processed data - Train: (525886, 30), Test: (538150, 29), Submission: (538150, 2)


,X344,X598,X385,X603,X674,X415,X345,X137,X174,X302,...,X333,label,volume_weighted_sell,buy_sell_ratio,selling_pressure,effective_spread_proxy,log_volume,bid_ask_imbalance,order_flow_imbalance,liquidity_ratio
0,-0.677325,0.274873,-0.784012,-1.159330,-0.260499,-1.167971,0.213585,0.671167,-0.691527,-0.617101,...,0.387715,0.562539,9958.962776,3.921505,0.203190,0.593620,5.404428,0.289269,0.593620,0.107088
1,-0.692942,0.266103,-0.782945,-1.150072,-0.259146,-1.162024,0.213159,0.670053,-0.690872,-0.612456,...,0.385304,0.533686,272947.922200,1.633316,0.379749,0.240501,6.743819,0.885843,0.240501,0.048273
2,-0.683432,0.257616,-0.781879,-1.140890,-0.257801,-1.156110,0.212734,0.668940,-0.690220,-0.618360,...,0.382907,0.546505,40310.130924,1.167619,0.461336,0.077329,5.692371,-0.985435,0.077329,0.205321
3,-0.670473,0.249403,-0.859988,-1.131785,-0.256463,-1.353974,0.212310,0.667829,-0.709082,-0.603181,...,0.380523,0.357703,57571.078915,2.686731,0.271243,0.457514,6.134926,-0.624049,0.457514,0.056177
4,-0.661079,0.241454,-0.858815,-1.122754,-0.255132,-1.346996,0.211886,0.666719,-0.708379,-0.596702,...,0.378152,0.362452,6342.118926,2.216115,0.310934,0.378132,4.968549,0.774511,0.378132,0.214322
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525881,1.781749,0.018458,-1.872166,0.723683,0.402747,-1.168157,2.053096,-0.372790,-0.123341,-1.877339,...,1.514873,0.396289,5224.470188,0.705263,0.586420,0.172840,4.557953,-0.240883,-0.172840,0.116201
525882,1.778885,0.017959,-1.870093,0.717379,0.400425,-1.162209,2.050117,-0.372456,-0.124263,-1.865351,...,1.506218,0.328993,11914.254612,1.640604,0.378701,0.242597,5.183871,-0.278513,0.242597,0.035789
525883,1.776025,0.017477,-1.867518,0.711127,0.398116,-1.166911,2.047143,-0.372121,-0.125182,-1.845242,...,1.497611,0.189909,3113.802756,2.292427,0.303727,0.392545,4.627440,0.179903,0.392545,0.087672
525884,1.773169,0.017010,-1.864947,0.704926,0.395820,-1.159929,2.044172,-0.371788,-0.124341,-1.861371,...,1.489051,0.410831,3891.659200,0.428489,0.700040,0.400080,4.324927,0.078066,-0.400080,0.142597


In [16]:
oof_preds, test_preds, model_slices = train_and_evaluate(train_df, test_df)


--- Fold 1/3 ---
  Training slice: full_data, samples: 350590
  Training slice: last_75pct, samples: 350590
  Training slice: last_50pct, samples: 262943

--- Fold 2/3 ---
  Training slice: full_data, samples: 350591
  Training slice: last_75pct, samples: 219120
  Training slice: last_50pct, samples: 175295

--- Fold 3/3 ---
  Training slice: full_data, samples: 350591
  Training slice: last_75pct, samples: 219120
  Training slice: last_50pct, samples: 87648


In [17]:
ensemble_and_submit(train_df, oof_preds, test_preds, submission_df)


XGB Simple Ensemble Pearson:   0.0542
XGB Weighted Ensemble Pearson: 0.0566

LGBM Simple Ensemble Pearson:   0.0183
LGBM Weighted Ensemble Pearson: 0.0212

FINAL ensemble across learners Pearson: 0.0352
Saved: submission.csv
